In [27]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [28]:
x = tf.random.normal((100000, 6))
y = tf.random.uniform((100000, 1), minval=0, maxval=2, dtype=tf.int32)

x = np.array(x).astype('float32') / 255.0
y = np.array(y)

print(x.shape)
print() 
print(y)
print() 
print(x)

(100000, 6)

[[0]
 [0]
 [0]
 ...
 [0]
 [0]
 [0]]

[[-1.1513449e-03  1.2896467e-03  2.5075674e-03  2.0111541e-03
   6.5337229e-03  5.3149336e-03]
 [-1.8963463e-03 -6.0651833e-03  2.7160873e-03  3.3179915e-03
   4.8860745e-03  2.0574552e-03]
 [ 1.0223159e-03 -4.4887760e-03  6.1047322e-05 -4.2000418e-03
   4.4352403e-03 -3.4909623e-03]
 ...
 [-3.0177999e-03  2.3794328e-03  1.3850330e-03  3.4065053e-03
  -1.1231254e-03 -1.0974342e-03]
 [-3.9124928e-04 -2.0199479e-03 -3.7397435e-03 -5.5416237e-04
   1.5311080e-03 -8.5707940e-03]
 [-1.8041955e-03  3.0742746e-03  7.5062294e-03 -2.6825690e-03
  -2.3719573e-03 -1.0717258e-02]]


In [29]:
from sklearn.model_selection import train_test_split as tts

x_train, x_test, y_train, y_test = tts(x, y, test_size=0.2, random_state=42)

print(x_train.shape)
print(x_test.shape)

(80000, 6)
(20000, 6)


In [30]:
import optuna
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping

In [31]:
def my_model(trial):
    lr = trial.suggest_float('lr', 1e-8, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1,15)


    model = keras.Sequential([
        keras.Input(shape=(6,))
    ])
    

    for i in range(n_layers):
        neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)
        activation_fn = trial.suggest_categorical(f'activation_fn_{i}', ['relu', 'leaky_relu', 'elu', 'tanh'])
        initializer = trial.suggest_categorical(f'initializer_{i}', ['he_normal', 'glorot_normal'])
        dropouts = trial.suggest_float(f'dropouts_{i}', 0.0, 0.4, step=0.1)

        regularizer = trial.suggest_categorical(f'regularizer_{i}', ['l1','l2'])
        reg_rate = trial.suggest_float(f'reg_rate_{i}', 1e-8, 1e-2, log=True)
        if regularizer == 'l1':
            chosen_regularizer = regularizers.l1(reg_rate)
        else:
            chosen_regularizer = regularizers.l2(reg_rate)


        model.add(layers.Dense(neurons, kernel_initializer=initializer, kernel_regularizer=chosen_regularizer, activation=activation_fn))

        if dropouts > 0.0:
            model.add(layers.Dropout(dropouts))


    model.compile(
        loss = keras.losses.BinaryCrossentropy(),
        optimizer = keras.optimizers.RMSprop(lr),
        metrics = ['accuracy']
    )

    early_stop = EarlyStopping(
        patience = 3,
        verbose = 1,
        restore_best_weights = True,
        monitor = 'val_loss'
    )

    history = model.fit(
        x_train, y_train,
        callbacks = [early_stop],
        epochs = 10,
        verbose = 1,
        batch_size = 128
    )

    trial.set_user_attr('trn_acc', history.history['accuracy'])
    trial.set_user_attr('trn_loss', history.history['loss'])
    trial.set_user_attr('val_acc', history.history['val_accuracy'])
    trial.set_user_attr('val_loss', history.history['val_loss'])

    model.save(f'model_trial_{trial.number}.keras')

    return max(history.history['val_accuracy'])



study = optuna.create_study(direction='maximize')
study.optimize(my_model, n_trials=5)

[I 2026-09-24 17:51:39,901] A new study created in memory with name: no-name-58a3d044-2242-4bc4-9047-00e9b0e690ae
/tmp/ipykernel_58/45686437.py:12: UserWarning: The distribution is specified by [1, 200] and step=16, but the range is not divisible by `step`. It will be replaced with [1, 193].
  neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)


Epoch 1/10


[W 2026-09-24 17:51:40,465] Trial 0 failed with parameters: {'lr': 2.2679601970317534e-07, 'n_layers': 12, 'neurons_0': 97, 'activation_fn_0': 'relu', 'initializer_0': 'he_normal', 'dropouts_0': 0.1, 'regularizer_0': 'l1', 'reg_rate_0': 0.00018460285428555995, 'neurons_1': 97, 'activation_fn_1': 'relu', 'initializer_1': 'glorot_normal', 'dropouts_1': 0.4, 'regularizer_1': 'l1', 'reg_rate_1': 0.002450108600410473, 'neurons_2': 129, 'activation_fn_2': 'tanh', 'initializer_2': 'glorot_normal', 'dropouts_2': 0.4, 'regularizer_2': 'l1', 'reg_rate_2': 1.1772827176467562e-05, 'neurons_3': 65, 'activation_fn_3': 'leaky_relu', 'initializer_3': 'glorot_normal', 'dropouts_3': 0.30000000000000004, 'regularizer_3': 'l1', 'reg_rate_3': 0.0008943099406705237, 'neurons_4': 65, 'activation_fn_4': 'leaky_relu', 'initializer_4': 'he_normal', 'dropouts_4': 0.30000000000000004, 'regularizer_4': 'l2', 'reg_rate_4': 0.0022176869148921047, 'neurons_5': 113, 'activation_fn_5': 'relu', 'initializer_5': 'he_norm

ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(128, 1), output.shape=(128, 17)